# **CROSS VALIDATION Y FINE TUNING**

### **Optimización Robusta de Hiperparámetros (Optuna + K-Fold)**

Una vez demostrada la viabilidad de la arquitectura *Deep Neurosymbolic Learning* (DNSL) en la fase base, el siguiente paso crítico es la selección de sus hiperparámetros óptimos. En arquitecturas complejas que combinan aprendizaje profundo y restricciones físicas, parámetros como el coeficiente de castigo físico () o la tasa de aprendizaje dictan la diferencia entre un modelo que ignora la física y uno que la interioriza correctamente.

Para garantizar la máxima robustez y evitar el antipatrón de *Data Leakage* por *cherry-picking* (sobreajustar los hiperparámetros a una partición de datos "fácil"), se ha diseñado un flujo de optimización avanzado que combina el framework **Optuna** con una validación cruzada estricta (**K-Fold Cross Validation**).

#### **Lógica de Ejecución del Script:**

1. **Espacio de Búsqueda Dinámico:** Optuna explora inteligentemente miles de combinaciones posibles para tres variables clave:
* **`lr` (Learning Rate):** Define la velocidad a la que la red neuronal actualiza sus pesos (escala logarítmica entre  y ).
* **`dropout_prob`:** Nivel de regularización de la capa densa final (entre  y ) para mitigar el sobreajuste.
* **`max_lambda` ():** El hiperparámetro neurosimbólico más importante. Define el peso máximo del castigo físico en la función de pérdida (entre  y ).


2. **Aislamiento Estadístico (K-Fold):** Por cada "intento" (combinación de parámetros) propuesto por Optuna, no se entrena un solo modelo, sino **tres modelos distintos ()**. El dataset se divide en 3 pliegues basados en el identificador único del ciclo (`Cycle_ID`). Esto garantiza que el escalado (`StandardScaler`) y el entrenamiento de cada pliegue sean completamente ciegos a su respectivo conjunto de validación.
3. **Curriculum Learning & Early Stopping:** Dentro de cada *fold*, el entrenamiento respeta las fases de maduración de la red. Primero,  épocas de aprendizaje estadístico puro, seguidas de una rampa de  épocas donde se introduce progresivamente el valor de  sugerido por Optuna. Para evitar un gasto computacional innecesario, se ha implementado una doble lógica de parada temprana:
* **Early Stopping Clásico:** Detiene el entrenamiento del *fold* si la pérdida de validación no mejora en 15 épocas consecutivas.
* **Optuna Pruning (Poda Inteligente):** Si durante las primeras épocas de un intento, Optuna detecta que el error promedio es catastrófico comparado con intentos anteriores, cancela inmediatamente la ejecución de ese *trial*, ahorrando recursos.


4. **Función Objetivo:** Una vez entrenados los  modelos de un intento, se calcula la media aritmética de sus pérdidas de validación (`mean_cv_loss`). Este es el valor que Optuna intentará **minimizar** en las iteraciones sucesivas.

> **Objetivo Final:** Este flujo garantiza que los hiperparámetros ganadores no sean aquellos que tuvieron "suerte" en una partición aleatoria, sino aquellos que demuestran estabilidad, convergencia rápida y respeto por las leyes termodinámicas independientemente del conjunto de datos con el que se evalúen.

In [1]:
# ==============================================================================
# 🚀 OPTIMIZACIÓN DE HIPERPARÁMETROS (OPTUNA + K-FOLD) - ESTRICTO SIN LEAKAGE
# ==============================================================================

import os
import pandas as pd
import numpy as np
import copy
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import optuna
from sklearn.metrics import accuracy_score, recall_score, f1_score

# ------------------------------------------------------------------------------
# 1. SETUP Y CARGA DE DATOS BASE (A 10Hz, SIN PREPROCESAR)
# ------------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Usando dispositivo: {device}")

# ATENCIÓN: Debes cargar los datos agrupados a 10Hz ANTES de la ingeniería de variables
ruta_datos = '../../data/processed/hydraulic_10hz_raw.csv'
if not os.path.exists(ruta_datos):
    raise FileNotFoundError(f"❌ No se encuentra el dataset en: {ruta_datos}")

df_base = pd.read_csv(ruta_datos)
print(f"📦 Dataset 10Hz base cargado: {df_base.shape[0]} filas.")

targets = ['Target_Fouling', 'Target_Valvula', 'Target_Bomba', 'Target_Acumulador']
cols_sensores_base = ['PS1', 'PS3', 'EPS1', 'FS1', 'TS1', 'TS2', 'VS1']

# Obtenemos IDs originales (filtramos aumentados si por error se colaron)
ids_originales = [cid for cid in df_base['Cycle_ID'].unique() if cid < 10000]

# ------------------------------------------------------------------------------
# 2. DEFINICIÓN DE ARQUITECTURA Y PÉRDIDA FÍSICA
# ------------------------------------------------------------------------------
class CNN_Pasteurizer(nn.Module):
    def __init__(self, n_sensors, n_classes=3, dropout_prob=0.5):
        super(CNN_Pasteurizer, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_sensors, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.1),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.1),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.dropout_final = nn.Dropout(dropout_prob)
        self.head_fouling = nn.Linear(256, n_classes)
        self.head_valvula = nn.Linear(256, n_classes)
        self.head_bomba = nn.Linear(256, n_classes)
        self.head_acumulador = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.features(x).squeeze(-1)
        x = self.dropout_final(x)
        return self.head_fouling(x), self.head_valvula(x), self.head_bomba(x), self.head_acumulador(x)

class PhysicsGuidedLoss(nn.Module):
    def __init__(self, feature_cols, scaler, lambda_phys=1.0):
        super(PhysicsGuidedLoss, self).__init__()
        self.lambda_phys = lambda_phys
        self.register_buffer('means', torch.tensor(scaler.mean_, dtype=torch.float32))
        self.register_buffer('scales', torch.tensor(scaler.scale_, dtype=torch.float32))
        
        try:
            self.idx_ps1 = feature_cols.index('PS1_rmean') 
            self.idx_eps1 = feature_cols.index('EPS1_rmean')
            self.idx_fs1 = feature_cols.index('FS1_rmean') 
            self.idx_ts1 = feature_cols.index('TS1_rmean') 
            self.idx_ts2 = feature_cols.index('TS2_rmean') 
        except ValueError:
            self.idx_ps1 = feature_cols.index('PS1')
            self.idx_eps1 = feature_cols.index('EPS1')
            self.idx_fs1 = feature_cols.index('FS1')
            self.idx_ts1 = feature_cols.index('TS1')
            self.idx_ts2 = feature_cols.index('TS2')

        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, preds, targets, inputs_tensor):
        inputs_mean = inputs_tensor.mean(dim=2) 
        X_real = inputs_mean * self.scales + self.means
        
        P_in_bar, Q_lmin, Power_W = X_real[:, self.idx_ps1], X_real[:, self.idx_fs1], X_real[:, self.idx_eps1]
        T_in, T_out = X_real[:, self.idx_ts1], X_real[:, self.idx_ts2] 

        # Regla 1: Bomba
        P_hyd_est = P_in_bar * Q_lmin * 1.6666
        efficiency = P_hyd_est / (torch.abs(Power_W) + 1.0)
        prob_sano_bomba = torch.softmax(preds[2], dim=1)[:, 0]
        violation_pump = torch.relu(0.75 - efficiency) ** 2 
        loss_physics_pump = torch.mean(violation_pump * prob_sano_bomba) * 100.0

        # Regla 2: Pasteurizador (Corregido a Delta T real del dataset hidráulico: 2.0 ºC)
        m_dot = Q_lmin * 0.017
        Heat_Transfer_kW = m_dot * 3.9 * (T_out - T_in)
        Q_expected = m_dot * 3.9 * 2.0 
        
        thermal_efficiency = Heat_Transfer_kW / (Q_expected + 0.1)
        prob_sano_cooler = torch.softmax(preds[0], dim=1)[:, 0]
        violation_cooler = torch.relu(0.80 - thermal_efficiency) ** 2
        loss_physics_cooler = torch.mean(violation_cooler * prob_sano_cooler) * 100.0

        supervised_loss = sum([self.ce_loss(preds[i], targets[:, i]) for i in range(4)])
        total_loss = supervised_loss + (self.lambda_phys * (loss_physics_pump + loss_physics_cooler))
        
        return total_loss, supervised_loss.item(), loss_physics_pump.item(), loss_physics_cooler.item()

# ------------------------------------------------------------------------------
# 3. PIPELINE DE PREPROCESAMIENTO AISLADO POR FOLD
# ------------------------------------------------------------------------------
def preprocesar_fold(df_input, train_cycles, val_cycles, cols_sensores_base, noise_level=0.20):
    """Ejecuta traslación, aumento e ingeniería de variables aislando Train y Val."""
    df_fold = df_input.copy()
    
    # 1. TRASLACIÓN TÉRMICA BASADA SOLO EN TRAIN
    mask_train_real = df_fold['Cycle_ID'].isin(train_cycles)
    ts1_mean_train = df_fold.loc[mask_train_real, 'TS1'].mean()
    offset_temperatura = 65.0 - ts1_mean_train
    df_fold['TS1'] += offset_temperatura
    df_fold['TS2'] += offset_temperatura
    
    # 2. DATA AUGMENTATION (Solo Train)
    df_train_only = df_fold[mask_train_real].copy()
    df_noisy = df_train_only.copy()
    df_noisy['Cycle_ID'] += 50000 
    
    for c_name in cols_sensores_base:
        std_dev = df_noisy[c_name].std()
        noise = np.random.normal(0, std_dev * noise_level, size=len(df_noisy))
        df_noisy[c_name] += noise
            
    df_augmented = pd.concat([df_fold, df_noisy], ignore_index=True)
    train_cycles_aug = np.concatenate([train_cycles, df_noisy['Cycle_ID'].unique()])
    
    # 3. FEATURE ENGINEERING
    def engineer_features(group):
        group = group.sort_values('Time_Segundos') 
        X = group[cols_sensores_base]                    
        rmean = X.rolling(5, min_periods=1).mean().add_suffix('_rmean')
        rstd = X.rolling(5, min_periods=1).std().fillna(0).add_suffix('_rstd')
        lag = X.shift(1).bfill().add_suffix('_lag1') 
        return pd.concat([group, rmean, rstd, lag], axis=1)

    df_final = df_augmented.groupby('Cycle_ID', group_keys=False).apply(engineer_features).reset_index(drop=True)
    
    # Obtener columnas finales
    cols_nuevas = [c for c in df_final.columns if '_rmean' in c or '_rstd' in c or '_lag1' in c]
    feature_cols = cols_sensores_base + cols_nuevas
    
    return df_final, feature_cols, train_cycles_aug

def create_tensors(df_in, cycle_ids, feature_cols):
    df_subset = df_in[df_in['Cycle_ID'].isin(cycle_ids)]
    target_subset = df_subset.groupby('Cycle_ID')[targets].first()
    
    x_list, y_list = [], []
    for cid in df_subset['Cycle_ID'].unique():
        cycle_data = df_subset[df_subset['Cycle_ID'] == cid][feature_cols].values
        if len(cycle_data) > 600: cycle_data = cycle_data[:600]
        elif len(cycle_data) < 600:
            pad = np.zeros((600 - len(cycle_data), len(feature_cols)))
            cycle_data = np.vstack([cycle_data, pad])
        x_list.append(cycle_data)
        y_list.append(target_subset.loc[cid].values)
    
    X_t = torch.tensor(np.array(x_list), dtype=torch.float32).permute(0, 2, 1)
    y_t = torch.tensor(np.array(y_list), dtype=torch.long)
    return X_t, y_t

# ------------------------------------------------------------------------------
# 4. FUNCIÓN OBJETIVO DE OPTUNA
# ------------------------------------------------------------------------------
def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    dropout_prob = trial.suggest_float("dropout_prob", 0.2, 0.6)
    max_lambda = trial.suggest_float("max_lambda", 1.0, 10.0)
    
    K_FOLDS = 5 
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state = 42)
    
    fold_val_losses, fold_val_metrics = [], []
    ciclos_numpy = np.array(ids_originales)
    
    EPOCHS_OPT = 300 
    WARMUP_EPOCHS = 10 
    RAMP_UP_EPOCHS = 80
    patience = 15 

    for fold_idx, (train_index, val_index) in enumerate(kf.split(ciclos_numpy)):
        
        train_cycles = ciclos_numpy[train_index]
        val_cycles = ciclos_numpy[val_index]
        
        # 1. Preprocesamiento aislado
        df_fold, feature_cols, train_cycles_aug = preprocesar_fold(df_base, train_cycles, val_cycles, cols_sensores_base)
        
        # 2. Escalado ajustado solo a Train
        scaler = StandardScaler()
        mask_train_aug = df_fold['Cycle_ID'].isin(train_cycles_aug)
        scaler.fit(df_fold.loc[mask_train_aug, feature_cols])
        df_fold[feature_cols] = scaler.transform(df_fold[feature_cols])

        # 3. Creación de Tensores
        X_train_pt, y_train_pt = create_tensors(df_fold, train_cycles_aug, feature_cols)
        X_val_pt, y_val_pt = create_tensors(df_fold, val_cycles, feature_cols)
        
        train_loader = DataLoader(TensorDataset(X_train_pt, y_train_pt), batch_size=32, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val_pt, y_val_pt), batch_size=32, shuffle=False)
        
        model_opt = CNN_Pasteurizer(n_sensors=X_train_pt.shape[1], n_classes=3, dropout_prob=dropout_prob).to(device)
        optimizer_opt = optim.Adam(model_opt.parameters(), lr=lr)
        
        criterion_opt = PhysicsGuidedLoss(feature_cols=feature_cols, scaler=scaler, lambda_phys=0.0).to(device)
        criterion_val = nn.CrossEntropyLoss()
        
        best_val_loss_fold = float('inf')
        best_metrics_fold = {} 
        counter = 0 
        
        for epoch in range(EPOCHS_OPT):
            if epoch < WARMUP_EPOCHS: current_lambda = 0.0
            elif epoch < (WARMUP_EPOCHS + RAMP_UP_EPOCHS):
                current_lambda = ((epoch - WARMUP_EPOCHS) / RAMP_UP_EPOCHS) * max_lambda
            else: current_lambda = max_lambda
            
            criterion_opt.lambda_phys = current_lambda
            
            model_opt.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer_opt.zero_grad()
                preds = model_opt(inputs)
                loss, _, _, _ = criterion_opt(preds, labels, inputs)
                loss.backward()
                optimizer_opt.step()
                
            model_opt.eval()
            val_loss = 0.0
            y_true_list, y_pred_list = [], []
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    preds = model_opt(inputs)
                    val_loss += sum([criterion_val(preds[i], labels[:, i]).item() for i in range(4)])
                    
                    preds_batch = torch.stack([p.argmax(1) for p in preds], dim=1).cpu().numpy()
                    y_true_list.append(labels.cpu().numpy())
                    y_pred_list.append(preds_batch)
                    
            avg_val_loss = val_loss / len(val_loader)
            
            if avg_val_loss < best_val_loss_fold:
                best_val_loss_fold = avg_val_loss
                counter = 0
                
                y_true_fold = np.vstack(y_true_list)
                y_pred_fold = np.vstack(y_pred_list)
                
                exact_match = (y_true_fold == y_pred_fold).all(axis=1).mean()
                recalls_m, f1s_m = [], []
                for i in range(4):
                    recalls_m.append(recall_score(y_true_fold[:, i], y_pred_fold[:, i], average='macro', zero_division=0))
                    f1s_m.append(f1_score(y_true_fold[:, i], y_pred_fold[:, i], average='macro', zero_division=0))
                
                best_metrics_fold = {
                    'exact_match': exact_match,
                    'recall_macro': np.mean(recalls_m),
                    'f1_macro': np.mean(f1s_m)
                }
            else:
                counter += 1
                
            trial.report(avg_val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            
            if counter >= patience and epoch > WARMUP_EPOCHS:
                break
                
        fold_val_losses.append(best_val_loss_fold)
        fold_val_metrics.append(best_metrics_fold)
        
    trial.set_user_attr('cv_exact_match', np.mean([m['exact_match'] for m in fold_val_metrics]))
    trial.set_user_attr('cv_recall_macro', np.mean([m['recall_macro'] for m in fold_val_metrics]))
    trial.set_user_attr('cv_f1_macro', np.mean([m['f1_macro'] for m in fold_val_metrics]))
        
    return np.mean(fold_val_losses)

# ------------------------------------------------------------------------------
# 5. EJECUCIÓN DEL ESTUDIO Y GUARDADO
# ------------------------------------------------------------------------------
print("\n⚙️ Lanzando Optuna...")
optuna.logging.set_verbosity(optuna.logging.INFO) 
study = optuna.create_study(direction="minimize", study_name="DNSL_Optimization")
study.optimize(objective, n_trials=30) 

print("\n" + "="*50)
print("🏆 OPTIMIZACIÓN COMPLETADA 🏆")
print("="*50)
print(f"Mejor valor (Val Loss Media): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"   > {key}: {value}")

best_trial = study.best_trial
print("\nMétricas de Validación Cruzada del Mejor Modelo:")
print(f"   > Exact Match:  {best_trial.user_attrs['cv_exact_match']:.4f}")
print(f"   > Recall Macro: {best_trial.user_attrs['cv_recall_macro']:.4f}")
print(f"   > F1-Score Macro: {best_trial.user_attrs['cv_f1_macro']:.4f}")

try:
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()
    fig2 = optuna.visualization.plot_param_importances(study)
    fig2.show()
except ImportError:
    pass

os.makedirs("../../models/fine_tuning/", exist_ok=True)
mejores_resultados = study.best_params.copy()
mejores_resultados['best_val_loss'] = study.best_value
mejores_resultados.update(best_trial.user_attrs) 

df_best_params = pd.DataFrame([mejores_resultados])
ruta_resultados = "../../models/fine_tuning/optuna_best_params_dns.csv"
df_best_params.to_csv(ruta_resultados, index=False)
print(f"\n💾 Mejores hiperparámetros guardados en: {ruta_resultados}")
df_todos_intentos = study.trials_dataframe()
df_todos_intentos.to_csv("../../models/fine_tuning/optuna_all_trials_history_dns.csv", index=False)

🖥️ Usando dispositivo: cuda


[I 2026-02-25 10:30:19,140] A new study created in memory with name: DNSL_Optimization


📦 Dataset 10Hz base cargado: 1325205 filas.

⚙️ Lanzando Optuna...


[I 2026-02-25 10:34:37,023] Trial 0 finished with value: 0.06183210644710561 and parameters: {'lr': 0.0022243234786004373, 'dropout_prob': 0.20219689010649033, 'max_lambda': 3.7585293696964697}. Best is trial 0 with value: 0.06183210644710561.
[I 2026-02-25 10:39:35,381] Trial 1 finished with value: 0.08235456328407216 and parameters: {'lr': 0.000503789627904275, 'dropout_prob': 0.3207898070540667, 'max_lambda': 5.2870584228156865}. Best is trial 0 with value: 0.06183210644710561.
[I 2026-02-25 10:44:39,904] Trial 2 finished with value: 0.06938594987166416 and parameters: {'lr': 0.0009035750848686588, 'dropout_prob': 0.5635209506683936, 'max_lambda': 3.1113156479649042}. Best is trial 0 with value: 0.06183210644710561.
[I 2026-02-25 10:48:31,543] Trial 3 finished with value: 0.07485078502572669 and parameters: {'lr': 0.003006243120036963, 'dropout_prob': 0.3571572911883245, 'max_lambda': 7.005986978458674}. Best is trial 0 with value: 0.06183210644710561.
[I 2026-02-25 10:52:24,998] Tr


🏆 OPTIMIZACIÓN COMPLETADA 🏆
Mejor valor (Val Loss Media): 0.0618
Mejores Hiperparámetros encontrados:
   > lr: 0.0022243234786004373
   > dropout_prob: 0.20219689010649033
   > max_lambda: 3.7585293696964697

Métricas de Validación Cruzada del Mejor Modelo:
   > Exact Match:  0.9873
   > Recall Macro: 0.9974
   > F1-Score Macro: 0.9967



💾 Mejores hiperparámetros guardados en: ../../models/fine_tuning/optuna_best_params_dns.csv


* **Mejor valor (Val Loss Media):** 0.0618

* **Mejores Hiperparámetros encontrados:**
   * lr: 0.0022243234786004373
   * dropout_prob: 0.20219689010649033
   * max_lambda: 3.7585293696964697

* **Métricas de Validación Cruzada del Mejor Modelo:**
   * Exact Match:  0.9873
   * Recall Macro: 0.9974
   * F1-Score Macro: 0.9967

## **Validación cruzada**

In [2]:
# ==============================================================================
# 🔄 VALIDACIÓN CRUZADA K-FOLD (k=5) - MODELO DNSL
# ==============================================================================

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ------------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Usando dispositivo: {device}")

ruta_datos = '../../data/processed/hydraulic_10hz_raw.csv'
if not os.path.exists(ruta_datos):
    raise FileNotFoundError(f"❌ No se encuentra el dataset en: {ruta_datos}")

df_base = pd.read_csv(ruta_datos)
targets = ['Target_Fouling', 'Target_Valvula', 'Target_Bomba', 'Target_Acumulador']
cols_sensores_base = ['PS1', 'PS3', 'EPS1', 'FS1', 'TS1', 'TS2', 'VS1']
ids_originales = np.array([cid for cid in df_base['Cycle_ID'].unique() if cid < 10000])

# Hiperparámetros estáticos 
lr = 0.002224
dropout_prob = 0.2021
max_lambda = 3.7585
EPOCHS = 300
WARMUP_EPOCHS = 10
RAMP_UP_EPOCHS = 80
patience = 15

# ------------------------------------------------------------------------------
# 2. ARQUITECTURA Y PÉRDIDA FÍSICA
# ------------------------------------------------------------------------------
class CNN_Pasteurizer(nn.Module):
    def __init__(self, n_sensors, n_classes=3, dropout_prob=0.5):
        super(CNN_Pasteurizer, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_sensors, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.1),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.1),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.dropout_final = nn.Dropout(dropout_prob)
        self.head_fouling = nn.Linear(256, n_classes)
        self.head_valvula = nn.Linear(256, n_classes)
        self.head_bomba = nn.Linear(256, n_classes)
        self.head_acumulador = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.features(x).squeeze(-1)
        x = self.dropout_final(x)
        return self.head_fouling(x), self.head_valvula(x), self.head_bomba(x), self.head_acumulador(x)

class PhysicsGuidedLoss(nn.Module):
    def __init__(self, feature_cols, scaler, lambda_phys=1.0):
        super(PhysicsGuidedLoss, self).__init__()
        self.lambda_phys = lambda_phys
        self.register_buffer('means', torch.tensor(scaler.mean_, dtype=torch.float32))
        self.register_buffer('scales', torch.tensor(scaler.scale_, dtype=torch.float32))
        
        try:
            self.idx_ps1 = feature_cols.index('PS1_rmean') 
            self.idx_eps1 = feature_cols.index('EPS1_rmean')
            self.idx_fs1 = feature_cols.index('FS1_rmean') 
            self.idx_ts1 = feature_cols.index('TS1_rmean') 
            self.idx_ts2 = feature_cols.index('TS2_rmean') 
        except ValueError:
            self.idx_ps1 = feature_cols.index('PS1')
            self.idx_eps1 = feature_cols.index('EPS1')
            self.idx_fs1 = feature_cols.index('FS1')
            self.idx_ts1 = feature_cols.index('TS1')
            self.idx_ts2 = feature_cols.index('TS2')

        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, preds, targets, inputs_tensor):
        inputs_mean = inputs_tensor.mean(dim=2) 
        X_real = inputs_mean * self.scales + self.means
        
        P_in_bar, Q_lmin, Power_W = X_real[:, self.idx_ps1], X_real[:, self.idx_fs1], X_real[:, self.idx_eps1]
        T_in, T_out = X_real[:, self.idx_ts1], X_real[:, self.idx_ts2] 

        # Regla 1: Bomba
        P_hyd_est = P_in_bar * Q_lmin * 1.6666
        efficiency = P_hyd_est / (torch.abs(Power_W) + 1.0)
        prob_sano_bomba = torch.softmax(preds[2], dim=1)[:, 0]
        violation_pump = torch.relu(0.75 - efficiency) ** 2 
        loss_physics_pump = torch.mean(violation_pump * prob_sano_bomba) * 100.0

        # Regla 2: Pasteurizador
        m_dot = Q_lmin * 0.017
        Heat_Transfer_kW = m_dot * 3.9 * (T_out - T_in)
        Q_expected = m_dot * 3.9 * 2.0 
        
        thermal_efficiency = Heat_Transfer_kW / (Q_expected + 0.1)
        prob_sano_cooler = torch.softmax(preds[0], dim=1)[:, 0]
        violation_cooler = torch.relu(0.80 - thermal_efficiency) ** 2
        loss_physics_cooler = torch.mean(violation_cooler * prob_sano_cooler) * 100.0

        supervised_loss = sum([self.ce_loss(preds[i], targets[:, i]) for i in range(4)])
        total_loss = supervised_loss + (self.lambda_phys * (loss_physics_pump + loss_physics_cooler))
        
        return total_loss

# ------------------------------------------------------------------------------
# 3. FUNCIONES AUXILIARES DE AISLAMIENTO
# ------------------------------------------------------------------------------
def preprocesar_fold(df_input, train_cycles, cols_sensores_base, noise_level=0.20):
    df_fold = df_input.copy()
    
    # Traslación térmica estricta basada solo en Train
    mask_train_real = df_fold['Cycle_ID'].isin(train_cycles)
    ts1_mean_train = df_fold.loc[mask_train_real, 'TS1'].mean()
    offset_temperatura = 65.0 - ts1_mean_train
    df_fold['TS1'] += offset_temperatura
    df_fold['TS2'] += offset_temperatura
    
    # Data Augmentation (Solo Train)
    df_train_only = df_fold[mask_train_real].copy()
    df_noisy = df_train_only.copy()
    df_noisy['Cycle_ID'] += 50000 
    
    for c_name in cols_sensores_base:
        std_dev = df_noisy[c_name].std()
        noise = np.random.normal(0, std_dev * noise_level, size=len(df_noisy))
        df_noisy[c_name] += noise
            
    df_augmented = pd.concat([df_fold, df_noisy], ignore_index=True)
    train_cycles_aug = np.concatenate([train_cycles, df_noisy['Cycle_ID'].unique()])
    
    # Feature Engineering
    def engineer_features(group):
        group = group.sort_values('Time_Segundos') 
        X = group[cols_sensores_base]                    
        rmean = X.rolling(5, min_periods=1).mean().add_suffix('_rmean')
        rstd = X.rolling(5, min_periods=1).std().fillna(0).add_suffix('_rstd')
        lag = X.shift(1).bfill().add_suffix('_lag1') 
        return pd.concat([group, rmean, rstd, lag], axis=1)

    df_final = df_augmented.groupby('Cycle_ID', group_keys=False).apply(engineer_features).reset_index(drop=True)
    cols_nuevas = [c for c in df_final.columns if '_rmean' in c or '_rstd' in c or '_lag1' in c]
    return df_final, cols_sensores_base + cols_nuevas, train_cycles_aug

def create_tensors(df_in, cycle_ids, feature_cols):
    df_subset = df_in[df_in['Cycle_ID'].isin(cycle_ids)]
    target_subset = df_subset.groupby('Cycle_ID')[targets].first()
    
    x_list, y_list = [], []
    for cid in df_subset['Cycle_ID'].unique():
        cycle_data = df_subset[df_subset['Cycle_ID'] == cid][feature_cols].values
        if len(cycle_data) > 600: cycle_data = cycle_data[:600]
        elif len(cycle_data) < 600:
            pad = np.zeros((600 - len(cycle_data), len(feature_cols)))
            cycle_data = np.vstack([cycle_data, pad])
        x_list.append(cycle_data)
        y_list.append(target_subset.loc[cid].values)
    
    X_t = torch.tensor(np.array(x_list), dtype=torch.float32).permute(0, 2, 1)
    y_t = torch.tensor(np.array(y_list), dtype=torch.long)
    return X_t, y_t

# ------------------------------------------------------------------------------
# 4. BUCLE DE VALIDACIÓN CRUZADA (K=5)
# ------------------------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = []

print("🚀 Iniciando K-Fold Cross Validation (k=5)...")

for fold, (train_index, val_index) in enumerate(kf.split(ids_originales), 1):
    print(f"\n--- Ejecutando Fold {fold}/5 ---")
    
    train_cycles = ids_originales[train_index]
    val_cycles = ids_originales[val_index]
    
    # Preprocesamiento y escalado aislado
    df_fold, feature_cols, train_cycles_aug = preprocesar_fold(df_base, train_cycles, cols_sensores_base)
    
    scaler = StandardScaler()
    mask_train_aug = df_fold['Cycle_ID'].isin(train_cycles_aug)
    scaler.fit(df_fold.loc[mask_train_aug, feature_cols])
    df_fold[feature_cols] = scaler.transform(df_fold[feature_cols])

    X_train_pt, y_train_pt = create_tensors(df_fold, train_cycles_aug, feature_cols)
    X_val_pt, y_val_pt = create_tensors(df_fold, val_cycles, feature_cols)
    
    train_loader = DataLoader(TensorDataset(X_train_pt, y_train_pt), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_pt, y_val_pt), batch_size=32, shuffle=False)
    
    model = CNN_Pasteurizer(n_sensors=X_train_pt.shape[1], n_classes=3, dropout_prob=dropout_prob).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = PhysicsGuidedLoss(feature_cols=feature_cols, scaler=scaler, lambda_phys=0.0).to(device)
    
    best_val_loss = float('inf')
    best_model_state = None
    counter = 0 
    
    # Entrenamiento
    for epoch in range(EPOCHS):
        if epoch < WARMUP_EPOCHS: current_lambda = 0.0
        elif epoch < (WARMUP_EPOCHS + RAMP_UP_EPOCHS): current_lambda = ((epoch - WARMUP_EPOCHS) / RAMP_UP_EPOCHS) * max_lambda
        else: current_lambda = max_lambda
        
        criterion.lambda_phys = current_lambda
        model.train()
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            preds = model(inputs)
            loss = criterion(preds, labels, inputs)
            loss.backward()
            optimizer.step()
            
        # Validación
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                preds = model(inputs)
                val_loss += criterion(preds, labels, inputs).item()
                
        avg_val_loss = val_loss / len(val_loader)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            
        if counter >= patience and epoch > WARMUP_EPOCHS:
            print(f"  ⏹️ Early Stopping en época {epoch+1}")
            break

    # --------------------------------------------------------------------------
    # 5. EXTRACCIÓN DE MÉTRICAS DEL FOLD
    # --------------------------------------------------------------------------
    model.load_state_dict(best_model_state)
    model.eval()
    y_true_list, y_pred_list = [], []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            preds = model(inputs)
            preds_batch = torch.stack([p.argmax(1) for p in preds], dim=1).cpu().numpy()
            y_true_list.append(labels.cpu().numpy())
            y_pred_list.append(preds_batch)
            
    y_true_fold = np.vstack(y_true_list)
    y_pred_fold = np.vstack(y_pred_list)
    
    exact_match = (y_true_fold == y_pred_fold).all(axis=1).mean()
    recalls_m, f1s_m = [], []
    for i in range(4):
        recalls_m.append(recall_score(y_true_fold[:, i], y_pred_fold[:, i], average='macro', zero_division=0))
        f1s_m.append(f1_score(y_true_fold[:, i], y_pred_fold[:, i], average='macro', zero_division=0))
    
    recall_macro = np.mean(recalls_m)
    f1_macro = np.mean(f1s_m)
    
    print(f"  ✅ Fold {fold} Completado | Exact Match: {exact_match:.4f} | Recall: {recall_macro:.4f} | F1: {f1_macro:.4f}")
    
    resultados_cv.append({
        'Fold': fold,
        'Exact_Match': exact_match,
        'Recall_Macro': recall_macro,
        'F1_Score_Macro': f1_macro
    })

# ------------------------------------------------------------------------------
# 6. GUARDAR RESULTADOS
# ------------------------------------------------------------------------------
df_resultados = pd.DataFrame(resultados_cv)
os.makedirs("../../models/fine_tuning/", exist_ok=True)
ruta_csv = "../../models/fine_tuning/cv_kfold_metrics.csv"
df_resultados.to_csv(ruta_csv, index=False)

print("\n" + "="*50)
print("🏆 VALIDACIÓN CRUZADA COMPLETADA 🏆")
print("="*50)
print(df_resultados.mean(numeric_only=True).to_frame(name='Media Global'))
print(f"\n💾 Métricas guardadas exitosamente en: {ruta_csv}")

🖥️ Usando dispositivo: cuda
🚀 Iniciando K-Fold Cross Validation (k=5)...

--- Ejecutando Fold 1/5 ---
  ⏹️ Early Stopping en época 57
  ✅ Fold 1 Completado | Exact Match: 0.9683 | Recall: 0.9915 | F1: 0.9912

--- Ejecutando Fold 2/5 ---
  ⏹️ Early Stopping en época 41
  ✅ Fold 2 Completado | Exact Match: 0.9773 | Recall: 0.9951 | F1: 0.9929

--- Ejecutando Fold 3/5 ---
  ⏹️ Early Stopping en época 43
  ✅ Fold 3 Completado | Exact Match: 0.9751 | Recall: 0.9950 | F1: 0.9939

--- Ejecutando Fold 4/5 ---
  ⏹️ Early Stopping en época 38
  ✅ Fold 4 Completado | Exact Match: 0.9705 | Recall: 0.9913 | F1: 0.9909

--- Ejecutando Fold 5/5 ---
  ⏹️ Early Stopping en época 41
  ✅ Fold 5 Completado | Exact Match: 0.9751 | Recall: 0.9945 | F1: 0.9936

🏆 VALIDACIÓN CRUZADA COMPLETADA 🏆
                Media Global
Fold                3.000000
Exact_Match         0.973243
Recall_Macro        0.993474
F1_Score_Macro      0.992491

💾 Métricas guardadas exitosamente en: ../../models/fine_tuning/cv_kfold